# Comparação de Modelos Whisper

Este notebook tem como objetivo comparar diferentes modelos do OpenAI Whisper em termos de:
- **Latência**: Tempo necessário para transcrever o áudio.
- **Qualidade**: O texto transcrito resultante.

Vamos testar os modelos: `tiny`, `base`, `small`, `medium` (e `large` opcionalmente).

In [1]:
import whisper
import time
import pandas as pd
import os
import sys

# Adicionar pasta src ao path para importar funções
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', 'src')))
from extrair_audio import baixar_audio_youtube, transcrever_arquivo

from IPython.display import display, Audio

## 1. Preparação do Áudio do podcast

Vamos garantir que temos um arquivo de áudio para os testes. Se o arquivo não existir, baixaremos um vídeo curto do YouTube.

In [ ]:
AUDIO_DIR = "../data"
AUDIO_FILE = "audio_podcast.mp3"
AUDIO_PATH = os.path.join(AUDIO_DIR, AUDIO_FILE)

# URL de um vídeo o para teste
# Substitua por outra URL se desejar testar com outro áudio
YOUTUBE_URL = "https://www.youtube.com/watch?v=4ZqWA9avIws" 

if not os.path.exists(AUDIO_PATH):
    print("⏳ Baixando áudio para teste...")
    try:
        baixar_audio_youtube(YOUTUBE_URL, AUDIO_PATH, quiet=False)
        print(f"✅ Áudio baixado com sucesso: {AUDIO_PATH}")
    except Exception as e:
        print(f"❌ Erro ao baixar áudio: {e}")
else:
    print(f"✅ Arquivo de áudio já existe: {AUDIO_PATH}")

⏳ Baixando áudio para teste...
⏳ Baixando áudio com yt-dlp...
[youtube] Extracting URL: https://www.youtube.com/watch?v=4ZqWA9avIws
[youtube] 4ZqWA9avIws: Downloading webpage
[youtube] 4ZqWA9avIws: Downloading android sdkless player API JSON
[youtube] 4ZqWA9avIws: Downloading tv client config
[youtube] 4ZqWA9avIws: Downloading player 217a23a9-main
[youtube] 4ZqWA9avIws: Downloading tv player API JSON
[youtube] 4ZqWA9avIws: Downloading web safari player API JSON
[youtube] 4ZqWA9avIws: Downloading m3u8 information
[info] 4ZqWA9avIws: Downloading 1 format(s): 251
[download] Sleeping 4.00 seconds as required by the site...
[download] Destination: ..\data\audio_teste.mp3
[download] 100% of  207.23MiB in 00:00:21 at 9.48MiB/s     
✅ Áudio baixado em: ../data\audio_teste.mp3
✅ Áudio baixado com sucesso: ../data\audio_teste.mp3


## 2. Configuração dos Modelos

Defina quais modelos deseja comparar. 
Modelos disponíveis: `tiny`, `base`, `small`, `medium`, `large`.

> **Nota**: Modelos maiores (`medium`, `large`) requerem mais VRAM (GPU) ou RAM (CPU) e levarão mais tempo.

In [3]:
modelos_para_testar = ["tiny", "base", "small", "medium"] # Adicione "large" se tiver hardware suficiente
resultados = []

## 3. Execução do Teste

Iteramos sobre os modelos, carregamos cada um, transcrevemos o áudio e medimos o tempo.

In [4]:
print(f"Iniciando testes com o arquivo: {AUDIO_PATH}")

for modelo_nome in modelos_para_testar:
    print(f"\n--- Testando modelo: {modelo_nome} ---")
    
    try:
        # Usando a função importada de extrair_audio.py
        # Ela retorna um dict com 'text', 'load_time', 'transcribe_time'
        resultado = transcrever_arquivo(AUDIO_PATH, modelo=modelo_nome)
        
        print(f"Modelo carregado em {resultado['load_time']:.2f}s")
        print(f"Transcrição concluída em {resultado['transcribe_time']:.2f}s")
        
        # Salvar Resultados
        resultados.append({
            "Modelo": modelo_nome,
            "Tempo Carregamento (s)": round(resultado['load_time'], 2),
            "Tempo Transcrição (s)": round(resultado['transcribe_time'], 2),
            "Texto": resultado["text"].strip()
        })
        
    except Exception as e:
        print(f"Erro ao testar modelo {modelo_nome}: {e}")
        resultados.append({
            "Modelo": modelo_nome,
            "Tempo Carregamento (s)": None,
            "Tempo Transcrição (s)": None,
            "Texto": f"Erro: {str(e)}"
        })

Iniciando testes com o arquivo: ../data\audio_teste.mp3

--- Testando modelo: tiny ---
⏳ Carregando modelo Whisper (tiny)...


100%|█████████████████████████████████████| 72.1M/72.1M [00:06<00:00, 10.8MiB/s]


⏳ Transcrevendo áudio...


c:\Users\HugoLeonardo\Documents\GitHub\Data-Science---Python\Mentoria_NLP\projeto_uv\.venv\Lib\site-packages\whisper\transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Transcrição concluída.
Modelo carregado em 10.34s
Transcrição concluída em 5982.56s

--- Testando modelo: base ---
⏳ Carregando modelo Whisper (base)...


100%|███████████████████████████████████████| 139M/139M [00:15<00:00, 9.61MiB/s]


⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 18.27s
Transcrição concluída em 12054.48s

--- Testando modelo: small ---
⏳ Carregando modelo Whisper (small)...
⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 6.26s
Transcrição concluída em 19922.09s

--- Testando modelo: medium ---
⏳ Carregando modelo Whisper (medium)...


100%|█████████████████████████████████████| 1.42G/1.42G [03:21<00:00, 7.60MiB/s]


⏳ Transcrevendo áudio...
✅ Transcrição concluída.
Modelo carregado em 221.60s
Transcrição concluída em 55881.25s


## 4. Análise dos Resultados

Abaixo apresentamos a tabela comparativa e os textos gerados.

In [5]:
df_resultados = pd.DataFrame(resultados)

# Exibir tabela de tempos
print("\n=== Comparação de Latência ===")
display(df_resultados[["Modelo", "Tempo Carregamento (s)", "Tempo Transcrição (s)"]])

# Exibir comparação de textos
print("\n=== Comparação de Qualidade (Texto) ===")
for index, row in df_resultados.iterrows():
    print(f"\n>>> Modelo: {row['Modelo']}")
    print(f"Tempo: {row['Tempo Transcrição (s)']}s")
    print("Texto:")
    print(row['Texto'][:500] + "..." if len(row['Texto']) > 500 else row['Texto'])
    print("-" * 80)


=== Comparação de Latência ===


,Modelo,Tempo Carregamento (s),Tempo Transcrição (s)
0,tiny,10.34,5982.56
1,base,18.27,12054.48
2,small,6.26,19922.09
3,medium,221.60,55881.25



=== Comparação de Qualidade (Texto) ===

>>> Modelo: tiny
Tempo: 5982.56s
Texto:
Porto que é estreiz de irmãos na área que falam com vocês, Rodrigo Chorroa, na minha frente meu brother, irmão Robert Andrade Fíriu Boat, na mesma peranotitur Pedro Iqs, a turmina até agora Ah, pai, tá, o caber de acordar e a seja acordou? Ah, faz de sacará-se, fez de fim. Bala no instagram do 3 irmãos e veque com o Robertinho, um moucumir, tá brincando, ele pegou pesabes Vê, John, no instagram, pode ter estreiz de irmãos, vejo-se, teve assim excesso nas brincadeiras pra mim, foi a mesma encadei...
--------------------------------------------------------------------------------

>>> Modelo: base
Tempo: 12054.48s
Texto:
Pode ter 3 irmãos na área, quem fala com vocês, Rodrigo Charró, na minha frente, meu brother, meu irmão Robert Andrade, filho, Borracha, na mesa operando nosso editor Pedre Hick, você tá dormindo até agora, Robert. Ah, pai, eu tava com a cabeça de acordar, e o que você já acordou? Ah, faz i

In [6]:
# Opcional: Salvar resultados em CSV
df_resultados.to_csv("../data/comparacao_whisper.csv", index=False)
print("Resultados salvos em data/comparacao_whisper.csv")

Resultados salvos em data/comparacao_whisper.csv
